# 03. Advanced: 사전학습 목표와 에이전트 루프 toy 구현

## 목표

이 노트북은 최신 VLM 논문에서 반복되는 세 가지 아이디어를 작은 알고리즘으로 재현합니다.

- 원본 시각 구조를 보존하는 visual pretraining 감각
- 경계 중심의 dense spatial perception 감각
- 월드 모델과 agentic harness의 닫힌 루프 감각

대형 모델 학습은 하지 않습니다. 대신 왜 이런 목표 함수와 실행 구조가 필요한지 이해하는 데 집중합니다.


## 1. Visual Pretraining toy objective

실제 visual pretraining은 이미지 패치나 잠재 벡터를 사용합니다. 여기서는 문서 페이지를 문자 격자로 보고, 일부 위치를 가린 뒤 주변 구조로 복원하는 toy objective를 만듭니다.


In [ ]:
from collections import Counter
from typing import List, Tuple

page = [
    list("TITLE....."),
    list(".........."),
    list("EQ..FIG..."),
    list("EQ..FIG..."),
    list("TABLE....."),
    list("ROW1......"),
    list("ROW2......"),
]

mask_positions = [(2, 0), (2, 4), (4, 0), (5, 0)]

def mask_grid(grid: List[List[str]], positions: List[Tuple[int, int]], mask_token: str = "?") -> List[List[str]]:
    # 원본을 훼손하지 않기 위해 각 row를 복사합니다.
    masked = [row[:] for row in grid]
    for r, c in positions:
        masked[r][c] = mask_token
    return masked

def neighbors(grid: List[List[str]], r: int, c: int) -> List[str]:
    # 4방향 이웃만 사용합니다. 실제 모델은 훨씬 넓은 attention context를 사용합니다.
    values = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]):
            if grid[nr][nc] != "?":
                values.append(grid[nr][nc])
    return values

def predict_masked_token(masked: List[List[str]], r: int, c: int) -> str:
    # 가장 흔한 주변 문자를 예측값으로 삼는 단순 baseline입니다.
    # 목적은 좋은 모델을 만드는 것이 아니라, '가려진 시각 구조 복원'이라는 학습 목표를 이해하는 것입니다.
    local = neighbors(masked, r, c)
    if not local:
        return "."
    return Counter(local).most_common(1)[0][0]

masked_page = mask_grid(page, mask_positions)
for row in masked_page:
    print("".join(row))

print("\n[predictions]")
for pos in mask_positions:
    r, c = pos
    print(pos, "target=", page[r][c], "pred=", predict_masked_token(masked_page, r, c))


## 2. Masked Boundary Modeling toy version

경계는 인접한 픽셀이나 패치의 값이 달라지는 위치입니다. 아래 예제는 작은 label map에서 경계를 찾고, 일부 경계를 가린 뒤 복원 후보를 계산합니다.


In [ ]:
label_map = [
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [0, 0, 2, 2, 1, 1],
    [0, 2, 2, 2, 2, 1],
    [2, 2, 2, 2, 2, 2],
]

def boundary_map(labels: List[List[int]]) -> List[List[int]]:
    """오른쪽이나 아래쪽 이웃과 label이 다르면 경계로 표시합니다."""
    h, w = len(labels), len(labels[0])
    result = [[0 for _ in range(w)] for _ in range(h)]
    for r in range(h):
        for c in range(w):
            here = labels[r][c]
            right_diff = c + 1 < w and labels[r][c + 1] != here
            down_diff = r + 1 < h and labels[r + 1][c] != here
            result[r][c] = 1 if right_diff or down_diff else 0
    return result

def show_binary(grid: List[List[int]]) -> None:
    for row in grid:
        print("".join("#" if value else "." for value in row))

bmap = boundary_map(label_map)
print("[boundary map]")
show_binary(bmap)


In [ ]:
def boundary_training_examples(labels: List[List[int]]) -> List[Tuple[Tuple[int, int], int, List[int]]]:
    """각 위치의 주변 label을 입력으로, boundary 여부를 target으로 만드는 데이터셋입니다.
    
    실제 Masked Boundary Modeling은 훨씬 복잡하지만, 핵심은 '의미 label만 맞히는 것'이 아니라
    '어디서 구조가 갈라지는지'를 직접 학습 신호로 만든다는 점입니다.
    """
    h, w = len(labels), len(labels[0])
    b = boundary_map(labels)
    examples = []
    for r in range(h):
        for c in range(w):
            local_labels = []
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nr, nc = r + dr, c + dc
                if 0 <= nr < h and 0 <= nc < w:
                    local_labels.append(labels[nr][nc])
            examples.append(((r, c), b[r][c], local_labels))
    return examples

examples = boundary_training_examples(label_map)
for position, target, local in examples[:10]:
    print(f"pos={position} local={local} -> boundary={target}")


## 3. 통합 생성 출력의 스키마 확장

통합 모델은 다양한 출력을 생성합니다. 고급 실무에서는 태스크마다 스키마를 만들고, 모델 출력이 스키마를 만족하는지 검사해야 합니다.


In [ ]:
import json

schemas = {
    "detect": {"required": ["task", "objects"]},
    "pose": {"required": ["task", "camera", "translation", "rotation"]},
    "caption": {"required": ["task", "text"]},
}

def validate_schema(payload: dict) -> Tuple[bool, str]:
    task = payload.get("task")
    if task not in schemas:
        return False, f"unknown task: {task}"
    missing = [key for key in schemas[task]["required"] if key not in payload]
    if missing:
        return False, f"missing keys: {missing}"
    return True, "ok"

outputs = [
    {"task": "caption", "text": "문서 오른쪽에 다이어그램이 있다."},
    {"task": "pose", "camera": "cam0", "translation": [0.1, 0.0, 1.2], "rotation": [0, 0, 0, 1]},
    {"task": "detect", "objects": [{"label": "formula", "bbox_xyxy": [12, 20, 160, 48]}]},
    {"task": "pose", "camera": "cam0"},
]

for output in outputs:
    ok, message = validate_schema(output)
    print(json.dumps(output, ensure_ascii=False), "->", ok, message)


## 4. Agentic world loop toy version

Infinite Worlds류 연구에서 중요한 점은 모델이 한 장면을 만드는 데서 끝나지 않고, 행동에 따라 다음 상태를 만들며 루프를 돈다는 것입니다. 아래 예제는 pilot agent와 director agent의 역할을 아주 작게 흉내 냅니다.


In [ ]:
world = {
    "position": 0,
    "energy": 3,
    "scene": ["start", "bridge", "tower", "gate"],
    "events": [],
}

def pilot_agent(state: dict) -> str:
    """캐릭터 행동을 선택하는 간단한 agent입니다."""
    if state["energy"] <= 0:
        return "rest"
    if state["position"] < len(state["scene"]) - 1:
        return "move_forward"
    return "inspect"

def director_agent(state: dict, action: str) -> str:
    """장면을 풍부하게 만드는 이벤트를 선택합니다."""
    location = state["scene"][state["position"]]
    if action == "move_forward":
        return f"{location} 뒤쪽에 새로운 경로가 열린다"
    if action == "rest":
        return f"{location}에서 조명이 안정된다"
    return f"{location}의 숨겨진 표식이 드러난다"

def step_world(state: dict) -> dict:
    # state를 직접 바꾸면 디버깅이 어려우므로 복사본을 만듭니다.
    next_state = {key: (value[:] if isinstance(value, list) else value) for key, value in state.items()}
    action = pilot_agent(next_state)
    event = director_agent(next_state, action)
    if action == "move_forward":
        next_state["position"] += 1
        next_state["energy"] -= 1
    elif action == "rest":
        next_state["energy"] += 1
    next_state["events"].append({"action": action, "event": event})
    return next_state

state = world
for t in range(6):
    state = step_world(state)
    print(f"t={t} position={state['position']} energy={state['energy']} last={state['events'][-1]}")


## 5. 연구 읽기에서 고급 체크포인트

- visual pretraining 논문을 읽을 때는 텍스트 경로와 이미지 경로가 같은 자료를 공정하게 비교했는지 확인합니다.
- long OCR 논문을 읽을 때는 메모리만 줄였는지, 정확도와 장거리 일관성도 유지했는지 봅니다.
- unified generation 논문을 읽을 때는 출력 스키마, 후처리, 평가 지표가 태스크별 전문 모델과 공정하게 비교되는지 봅니다.
- dense spatial perception 논문을 읽을 때는 semantic benchmark뿐 아니라 depth, normal, pose, reconstruction 같은 공간 지표를 봅니다.
- world model 논문을 읽을 때는 데모 품질뿐 아니라 상호작용 지연, 상태 누적, 실패 복구를 확인합니다.
